In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.research_data import write_json
from src.research_data import read_series
from src.backtest import run_backtest
from src.research_validation import portfolio_metrics


# 07 Walk Forward Out of Sample Backtest

Run the full daily strategy with frozen formation parameters, lagged next-close instructions, exact forecast horizons and a 5% equity budget per entry limited by available cash. There is no open-position count limit; only one position per pair is held. This is the only notebook that runs the main portfolio simulation.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Settings


In [ ]:
cfg = ResearchConfig().validate()


## 2. Load frozen formation data and settings

Module 04 and Module 06 previews are not fed in as a replacement for daily forecasting. The engine uses their shared functions across the complete test period.


In [ ]:
train_prices = pd.read_parquet("train_prices.parquet")
test_prices = pd.read_parquet("test_prices.parquet")
eligible_pairs = pd.read_parquet("eligible_pairs.parquet")
cointegrated_pairs = pd.read_parquet("cointegrated_pairs.parquet")
risk_free_rates = read_series("data/inputs/risk_free_rates.parquet")
display(pd.Series(cfg.to_dict()))


## 3. Run the simulation

This cell may take time. Diagnostics in Modules 08–11 read the equity and trade files produced below.


In [ ]:
result = run_backtest(
    train_prices, test_prices, eligible_pairs, cointegrated_pairs, risk_free_rates, config=cfg
)
trades = result["trades"]
equity_curve = result["equity_curve"]
summary = portfolio_metrics(result, cfg.initial_capital)
for name, frame in result.items():
    frame.to_parquet(name + ".parquet")
write_json("backtest_summary.json", summary)
display(pd.Series(summary, name="Backtest results"))


## 4. Accounting and timing checks

These checks fail if funding, timing or cash reconciliation differs from the declared methodology.


In [ ]:
assert equity_curve.cash.min() >= -1e-07
assert cfg.max_open_pairs is None or equity_curve.n_open_positions.max() <= cfg.max_open_pairs
assert np.allclose(equity_curve.equity, equity_curve.cash + equity_curve.open_position_value)
assert np.isclose(equity_curve.equity.iloc[-1], cfg.initial_capital + trades.pnl.sum())
if not trades.empty:
    assert (trades.entry_date > trades.signal_date).all()
    assert (trades.entry_premium <= trades.entry_budget + 1e-07).all()
display(trades.head(10))
equity_curve.equity.plot(figsize=(11, 4), title="Synthetic option portfolio")
plt.ylabel("Model equity")
plt.show()
